# ChordSense Pro — Training on Colab
GPU: L4 24GB | Runtime: GPU

> **Luôn chạy cell Setup đầu tiên** sau mỗi lần restart runtime.
> v3 dataset phải có trên Drive tại `MyDrive/ChordSensePro/DE/data/export/v3/`

## 0. Setup — chạy đầu tiên sau mỗi restart

In [ ]:
import os, yaml
from pathlib import Path

# ── Mount Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

# ── Config ───────────────────────────────────────────────────
DRIVE_PROJECT = "/content/drive/MyDrive/ChordSensePro"
REPO_DIR      = "/content/ChordSensePro"
DATA_V2       = "/content/data_v2"
DATA_DST      = "/content/data_v3"
P1_RUN_NAME   = "v3_5s_p1"  # hoặc "v3_3s_p1"
P2_RUN_NAME   = "v3_5s_p2"  # hoặc "v3_3s_p2"

# ── Clone / pull repo ─────────────────────────────────────────
if not os.path.exists(REPO_DIR):
    !git clone --branch qkhenh https://github.com/qkhenh/ChordSensePro.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin qkhenh

os.chdir(os.path.join(REPO_DIR, "AI"))
print("cwd:", os.getcwd())

# ── Install deps ──────────────────────────────────────────────
!pip install -q -r requirements.txt

# ── Build P1 config ───────────────────────────────────────────
cfg_p1 = yaml.safe_load(Path(f"configs/{P1_RUN_NAME}.yaml").read_text())
cfg_p1["data_dir"]       = DATA_DST
cfg_p1["checkpoint_dir"] = os.path.join(DRIVE_PROJECT, "AI", "results", P1_RUN_NAME)
cfg_p1["results_csv"]    = os.path.join(DRIVE_PROJECT, "AI", "results", P1_RUN_NAME, "results.csv")
os.makedirs(cfg_p1["checkpoint_dir"], exist_ok=True)
colab_cfg = f"configs/colab_{P1_RUN_NAME}.yaml"
Path(colab_cfg).write_text(yaml.dump(cfg_p1, default_flow_style=False))

# ── Build P2 config ───────────────────────────────────────────
cfg_p2 = yaml.safe_load(Path(f"configs/{P2_RUN_NAME}.yaml").read_text())
cfg_p2["data_dir"]       = DATA_DST
cfg_p2["checkpoint_dir"] = os.path.join(DRIVE_PROJECT, "AI", "results", P2_RUN_NAME)
cfg_p2["results_csv"]    = os.path.join(DRIVE_PROJECT, "AI", "results", P2_RUN_NAME, "results.csv")
os.makedirs(cfg_p2["checkpoint_dir"], exist_ok=True)
colab_p2_cfg = f"configs/colab_{P2_RUN_NAME}.yaml"
Path(colab_p2_cfg).write_text(yaml.dump(cfg_p2, default_flow_style=False))

P1_BEST  = os.path.join(cfg_p1["checkpoint_dir"], "best.pt")
P2_BEST  = os.path.join(cfg_p2["checkpoint_dir"], "best.pt")
OUT_ONNX = os.path.join(DRIVE_PROJECT, "AI", "results", f"model_{P2_RUN_NAME}.onnx")

print("P1 checkpoint_dir:", cfg_p1["checkpoint_dir"])
print("P2 checkpoint_dir:", cfg_p2["checkpoint_dir"])
print("P1 best.pt exists :", os.path.exists(P1_BEST))
print("P2 best.pt exists :", os.path.exists(P2_BEST))
print("\nSetup done. Run cells below in order.")


In [ ]:
!nvidia-smi

## 1. Prepare Data v3
Copy v3 từ Drive về /content. **Skip nếu audio đã có.**

In [ ]:
def _audio_ok(d):
    p = os.path.join(d, "train", "audio")
    return os.path.isdir(p) and len(os.listdir(p)) > 0

DATA_V3_SRC = os.path.join(DRIVE_PROJECT, "DE", "data", "export", "v3")
DATA_V2_SRC = os.path.join(DRIVE_PROJECT, "DE", "data", "export", "v2")

if _audio_ok(DATA_DST):
    print("v3 audio already at", DATA_DST)
elif _audio_ok(DATA_V3_SRC):
    print("Copying v3 from Drive...")
    !rm -rf {DATA_DST}
    !cp -r {DATA_V3_SRC} {DATA_DST}
else:
    print("v3 audio missing on Drive — rebuilding from v2")
    if not os.path.exists(DATA_V2):
        print("Copying v2 from Drive...")
        !cp -r {DATA_V2_SRC} {DATA_V2}
    !python ../DE/scripts/export_dataset_v3.py --v2-dir {DATA_V2} --out {DATA_DST}

import pandas as pd
df = pd.read_parquet(os.path.join(DATA_DST, "train", "chords.parquet"))
n_audio = len(os.listdir(os.path.join(DATA_DST, "train", "audio")))
print("train rows:", len(df), "| bass_id:", "bass_id" in df.columns)
print("train audio files:", n_audio)


---
## Phase 1 — Train Head Only (MERT frozen)
Chạy lần đầu hoặc nếu P1 chưa xong.

In [ ]:
!python -m src.train --config {colab_cfg}

In [ ]:
# Resume Phase 1 nếu bị disconnect
P1_LAST = os.path.join(cfg_p1["checkpoint_dir"], "last.pt")
if os.path.exists(P1_LAST) and not os.path.exists(P1_BEST):
    !python -m src.train --config {colab_cfg} --resume {P1_LAST}
else:
    print("P1 best.pt:", os.path.exists(P1_BEST), "| last.pt:", os.path.exists(P1_LAST))

---
## Phase 2 — LoRA Fine-tune MERT
Cần `best.pt` từ Phase 1 trên Drive.

In [ ]:
print("P1 best.pt:", os.path.exists(P1_BEST))
assert os.path.exists(P1_BEST), "Phase 1 chưa xong hoặc best.pt chưa sync về Drive!"
!python -m src.train --config {colab_p2_cfg} --phase1_ckpt {P1_BEST}

In [ ]:
# Resume Phase 2 nếu bị disconnect
P2_LAST = os.path.join(cfg_p2["checkpoint_dir"], "last.pt")
if os.path.exists(P2_LAST) and not os.path.exists(P2_BEST):
    !python -m src.train --config {colab_p2_cfg} --resume {P2_LAST}
else:
    print("P2 best.pt:", os.path.exists(P2_BEST), "| last.pt:", os.path.exists(P2_LAST))

---
## Export ONNX

In [ ]:
print("P2 best.pt:", os.path.exists(P2_BEST))
assert os.path.exists(P2_BEST), "Phase 2 chưa xong!"
!python -m src.export_onnx --config {colab_p2_cfg} --checkpoint {P2_BEST} --output {OUT_ONNX} --benchmark
print("ONNX saved to:", OUT_ONNX)

---
## Quick Inference Test

In [ ]:
test_audio_dir = os.path.join(DATA_DST, "test", "audio")
test_files = sorted(os.listdir(test_audio_dir))[:3]
for f in test_files:
    audio_path = os.path.join(test_audio_dir, f)
    !python -m src.infer --config {colab_p2_cfg} --checkpoint {P2_BEST} --audio {audio_path}